In [1]:
from datasets import load_dataset
from dataclasses import dataclass

from transformers import AutoModelForCausalLM, AutoTokenizer




import tiktoken

import torch

tokenizer = tiktoken.get_encoding("gpt2")
tokenizer.pad_token = tokenizer.eot_token

In [2]:
dataset = load_dataset("Rowan/hellaswag", split="validation")

print(dataset)

Dataset({
    features: ['ind', 'activity_label', 'ctx_a', 'ctx_b', 'ctx', 'endings', 'source_id', 'split', 'split_type', 'label'],
    num_rows: 10042
})


In [3]:
dataset[0]

{'ind': 24,
 'activity_label': 'Roof shingle removal',
 'ctx_a': 'A man is sitting on a roof.',
 'ctx_b': 'he',
 'ctx': 'A man is sitting on a roof. he',
 'endings': ['is using wrap to wrap a pair of skis.',
  'is ripping level tiles off.',
  "is holding a rubik's cube.",
  'starts pulling up roofing on a roof.'],
 'source_id': 'activitynet~v_-JhWjGDPHMY',
 'split': 'val',
 'split_type': 'indomain',
 'label': '3'}

In [4]:
for i in range(5):
    curr_data = dataset[i]
    context = curr_data["ctx"]
    endings = curr_data["endings"]
    label = curr_data["label"]

    print(f"""
    context: {context}
    endings: {endings}
    label: {label}
    """)


    context: A man is sitting on a roof. he
    endings: ['is using wrap to wrap a pair of skis.', 'is ripping level tiles off.', "is holding a rubik's cube.", 'starts pulling up roofing on a roof.']
    label: 3
    

    context: A lady walks to a barbell. She bends down and grabs the pole. the lady
    endings: ['swings and lands in her arms.', 'pulls the barbell forward.', 'pulls a rope attached to the barbell.', 'stands and lifts the weight over her head.']
    label: 3
    

    context: Two women in a child are shown in a canoe while a man pulls the canoe while standing in the water, with other individuals visible in the background. the child and a different man
    endings: ['are then shown paddling down a river in a boat while a woman talks.', 'are driving the canoe, they go down the river flowing side to side.', 'sit in a canoe while the man paddles.', 'walking go down the rapids, while the man in his helicopter almost falls and goes out of canoehood.']
    label: 2
    

  

In [5]:
dataset[0]['ctx'] + " " + dataset[0]['endings'][0]
tokenizer.encode("hello world" + " " + "Java")

[31373, 995, 7349]

In [6]:
dataset[0]

{'ind': 24,
 'activity_label': 'Roof shingle removal',
 'ctx_a': 'A man is sitting on a roof.',
 'ctx_b': 'he',
 'ctx': 'A man is sitting on a roof. he',
 'endings': ['is using wrap to wrap a pair of skis.',
  'is ripping level tiles off.',
  "is holding a rubik's cube.",
  'starts pulling up roofing on a roof.'],
 'source_id': 'activitynet~v_-JhWjGDPHMY',
 'split': 'val',
 'split_type': 'indomain',
 'label': '3'}

In [7]:
text = dataset[0]['ctx'] + " " + dataset[0]['endings'][0]
encoded = tokenizer.encode(text)
print(f"text: {text}")
print(f"encoded: {encoded}")
print(f"token lenght: {len(encoded)}")

text: A man is sitting on a roof. he is using wrap to wrap a pair of skis.
encoded: [32, 582, 318, 5586, 319, 257, 9753, 13, 339, 318, 1262, 14441, 284, 14441, 257, 5166, 286, 1341, 271, 13]
token lenght: 20


In [8]:
tokenizer.encode(dataset[0]['ctx'])

[32, 582, 318, 5586, 319, 257, 9753, 13, 339]

AttributeError: 'Encoding' object has no attribute 'eos_token'

In [38]:
import tiktoken
import torch


    
def encode_single(tokenizer, data, eot_token=50256) -> tuple[torch.Tensor, torch.Tensor]:
    ctx = data['ctx']
    ctx_enc = tokenizer.encode(ctx)
    
    ending_tokens = []
    max_seq = 0
    for ending in data['endings']:
        encoded = tokenizer.encode(" " + ending)
        ending_tokens.append(encoded)
        max_seq = max(max_seq, len(encoded) + len(ctx_enc))

    encoded_data = torch.full((4, max_seq), eot_token, dtype=torch.int32)
    mask = torch.full((4, max_seq), 0, dtype = torch.int8)

    for i, ending in enumerate(ending_tokens):
        curr_ctx_enc = ctx_enc.copy()
        curr_ctx_enc.extend(ending)
        size = len(curr_ctx_enc)
        encoded_data[i, :size] = torch.tensor(curr_ctx_enc)
        mask[i, len(ctx_enc):len(ctx_enc) + len(ending)] = torch.full((1, len(ending)), 1)
    return encoded_data, mask

tokenizer = tiktoken.get_encoding("gpt2")
encode_single(tokenizer, dataset[0])

(tensor([[   32,   582,   318,  5586,   319,   257,  9753,    13,   339,   318,
           1262, 14441,   284, 14441,   257,  5166,   286,  1341,   271,    13],
         [   32,   582,   318,  5586,   319,   257,  9753,    13,   339,   318,
          34759,  1241, 19867,   572,    13, 50256, 50256, 50256, 50256, 50256],
         [   32,   582,   318,  5586,   319,   257,  9753,    13,   339,   318,
           4769,   257,  6437,  1134,   338, 23441,    13, 50256, 50256, 50256],
         [   32,   582,   318,  5586,   319,   257,  9753,    13,   339,  4940,
          10427,   510,  9753,   278,   319,   257,  9753,    13, 50256, 50256]],
        dtype=torch.int32),
 tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
         [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0]],
        dtype=torch.int8))

In [ ]:
dataset[:5]

In [ ]:
!pip install pandas